# Risultati di RTRwRuleCard

Questo notebook mostra i risultati della tesi e, soprattutto, come sono calcolati.
Ogni numero qui dentro viene ricavato sul momento dai file del repository: non c'e'
niente scritto a mano, quindi quello che leggete e' sempre allineato con le tabelle
pubblicate.

Per eseguirlo servono solo pandas e scipy. Non servono i dataset e non viene
addestrato nessun modello: si leggono i risultati gia' salvati, quindi gira in
pochi secondi.

Le tabelle pronte stanno in questa stessa cartella, in formato CSV. Chi vuole solo
quelle puo' aprire il README accanto a questo file.

In [1]:
from pathlib import Path

import pandas as pd
from scipy.stats import wilcoxon

# Le tabelle gia' calcolate stanno in questa cartella.
# I risultati completi di ogni esecuzione stanno una cartella piu' in su.
QUI = Path.cwd()
RUN = QUI.parent / "risultati"

print("tabelle:", QUI)
print("esecuzioni:", RUN, "(trovata)" if RUN.exists() else "(non trovata)")

tabelle: C:\Users\manzo\Desktop\fork\RuleTreeRank-DS2026\risultati_spiegati
esecuzioni: C:\Users\manzo\Desktop\fork\RuleTreeRank-DS2026\risultati (trovata)


## 1. Come e' fatto il modello

RuleTreeRank lavora in due stadi.

Nel **primo stadio** un albero di regole poco profondo divide i documenti e da' a
ognuno un punteggio di base. Le foglie dell'albero, incrociate con la query,
definiscono quelle che chiamiamo celle: una cella e' l'insieme dei documenti di una
certa query che finiscono in una certa foglia. Su FINDHR una query e' un annuncio di
lavoro e i documenti sono i candidati, quindi una cella e' un gruppo di candidati
dello stesso annuncio che l'albero ha messo insieme.

Nel **secondo stadio** quel punteggio viene corretto guardando dentro la cella. Qui
ci sono due pezzi distinti:

1. la distanza appresa, che decide quali documenti della cella sono vicini fra loro.
   Nell'originale e' il PDT, un albero di distanza a coppie. Al suo posto abbiamo
   messo la GAM di RuleCard, che e' un modello additivo.
2. l'aggregatore, che decide come si combinano i vicini. E' un kNN: prende i cinque
   piu' vicini e fa la media delle loro etichette.

Una precisazione che conta, perche' e' facile fraintenderla. Fra gli esperimenti c'e'
anche una foresta dentro la cella, ma **non e' un aggregatore**: non prende i vicini e
non li combina. Sostituisce tutti e due i pezzi insieme. Non calcola nessuna distanza
e non cerca nessun vicino, addestra una random forest sui documenti della cella, dalle
caratteristiche al residuo da correggere, e in predizione risponde direttamente lei.
Per questo non e' interpretabile, e infatti non la proponiamo come modello: serve solo
a misurare quanto si lascia sul tavolo restando leggibili.

## 2. Da dove vengono i numeri

Ogni esecuzione salva tre file: la configurazione con cui e' stata lanciata, le
metriche complessive e l'NDCG@10 di ogni singola query di test. E' quest'ultimo file
che rende possibile il confronto appaiato di cui parliamo piu' avanti.

Qui sotto apriamo il file di una esecuzione: RTR con il PDT, dataset FINDHR, dieci
query per modello, primo seme.

In [2]:
grezzo = pd.read_csv(RUN / "FINDHR" / "rtr" / "phi10" / "seed0" / "ndcg_per_query.csv")

print("righe nel file:", len(grezzo), "(una per query di test)")
grezzo.head()

righe nel file: 100 (una per query di test)


,query,completo,solo_r,knn_euclideo
0,0,0.927894,0.921190,0.919316
1,1,0.848945,0.888206,0.949108
2,2,0.929969,0.938931,0.994830
3,3,0.836178,0.870815,0.836178
4,4,0.981706,0.932947,0.949855


Le colonne sono quattro. `completo` e' il modello intero ed e' quella che usiamo in
tutti i confronti. `solo_r` e' il punteggio del solo primo stadio, senza correzione.
`knn_euclideo` e' la correzione fatta con la distanza euclidea invece di quella
appresa. Le ultime due vengono salvate dalla stessa esecuzione, quindi certe analisi
si possono fare senza rilanciare niente.

## 3. Il confronto fra due modelli, spiegato e ricalcolato

Per dire se un modello e' meglio di un altro non confrontiamo due medie separate.
Confrontiamo i due modelli **sulla stessa query**, una per una, e poi guardiamo le
differenze. In questo modo la difficolta' della singola query non sporca il risultato:
se un annuncio e' difficile per tutti, lo e' per entrambi i modelli e la differenza
non ne risente.

Prima una funzione che legge l'NDCG per query di un modello, mediando sui semi.

In [3]:
def ndcg_per_query(modello, dataset, phi):
    """NDCG@10 di ogni query, mediato sulle cinque esecuzioni.

    Ogni esecuzione usa lo stesso codice e gli stessi dati, e cambia solo il seme
    del generatore casuale. Serve perche' gli alberi scelgono a caso quando due
    divisioni sono ugualmente buone: mediando sui semi si toglie quella
    variabilita', che da sola vale circa 0.0035 di NDCG.
    """
    serie = []
    for cartella in sorted((RUN / dataset / modello / f"phi{phi}").glob("seed*")):
        valori = pd.read_csv(cartella / "ndcg_per_query.csv").set_index("query")["completo"]
        serie.append(valori)
    return pd.concat(serie, axis=1).mean(axis=1)


pdt = ndcg_per_query("rtr", "FINDHR", 10)
gam = ndcg_per_query("rtrwrulecard", "FINDHR", 10)

print("query lette:", len(pdt), "| semi mediati:", len(list((RUN / "FINDHR" / "rtr" / "phi10").glob("seed*"))))

query lette: 100 | semi mediati: 5


Vediamo il conto in piccolo, sulle prime tre query, prima di farlo su tutte.

In [4]:
esempio = pd.DataFrame({"PDT": pdt, "GAM": gam})
esempio["differenza"] = esempio["GAM"] - esempio["PDT"]
esempio.head(3).round(4)

,PDT,GAM,differenza
query,,,
0,0.9279,0.9223,-0.0056
1,0.8489,0.9589,0.1100
2,0.9300,0.9409,0.0109


Sulla prima query i due modelli danno un certo NDCG, e la differenza dice quale dei
due ha ordinato meglio i candidati di quell'annuncio. Ripetendo per tutte le query e
facendo la media delle differenze si ottiene il guadagno complessivo, quello che
chiamiamo gain.

Il p viene da un test di Wilcoxon appaiato sulle stesse differenze. Il test scarta le
query dove i due modelli danno esattamente lo stesso valore, perche' non portano
informazione sul segno; la media invece le tiene, ed e' giusto cosi'.

In [5]:
differenze = (gam - pdt).dropna()
gain = differenze.mean()
p = wilcoxon(differenze[differenze != 0]).pvalue

print("query confrontate:", len(differenze))
print("query dove va meglio la GAM:", int((differenze > 0).sum()))
print("query dove va meglio il PDT:", int((differenze < 0).sum()))
print()
print("gain, cioe' la media delle differenze: %+.4f" % gain)
print("p del test di Wilcoxon appaiato:      %.4f" % p)

query confrontate: 100
query dove va meglio la GAM: 47
query dove va meglio il PDT: 27

gain, cioe' la media delle differenze: +0.0077
p del test di Wilcoxon appaiato:      0.0017


Lo stesso numero si trova nella tabella pubblicata, nelle colonne `gain` e
`p_wilcoxon`. Il confronto qui sotto serve a mostrare che le tabelle non contengono
niente che non sia ricavabile dai file delle esecuzioni.

In [6]:
pubblicata = pd.read_csv("confronto_multiseed_server.csv")
riga = pubblicata[(pubblicata["dataset"] == "FINDHR") & (pubblicata["phi"] == 10)].iloc[0]

print("nella tabella pubblicata: gain %+.4f, p %.4f" % (riga["gain"], riga["p_wilcoxon"]))
print("ricalcolato qui sopra:    gain %+.4f, p %.4f" % (gain, p))

nella tabella pubblicata: gain +0.0077, p 0.0017
ricalcolato qui sopra:    gain +0.0077, p 0.0017


Nelle tabelle c'e' anche una colonna `differenza`, che e' la media del modello A meno
la media del modello B. Viene identica al gain, e non per caso: l'NDCG complessivo che
salviamo e' la media semplice di quelli per query, quindi fare prima le differenze e
poi la media, o prima le medie e poi la differenza, e' la stessa operazione.

In [7]:
pubblicata["scarto"] = (pubblicata["differenza"] - pubblicata["gain"]).abs()
print("scarto massimo fra le due colonne su tutte le righe:", pubblicata["scarto"].max())

scarto massimo fra le due colonne su tutte le righe: 0.0


## 4. Risultato principale: la GAM contro il PDT

Stessa architettura, stessi iperparametri, stessi semi. Cambia solo il modello di
distanza. Dieci combinazioni, cioe' due dataset per cinque valori di query per modello.

In [8]:
principale = pd.read_csv("confronto_multiseed_server.csv")
principale[["dataset", "phi", "pdt_media", "gam_media", "gain", "p_wilcoxon"]]

,dataset,phi,pdt_media,gam_media,gain,p_wilcoxon
0,FINDHRLIST,1,0.9770,0.9784,0.0014,0.0161
1,FINDHRLIST,2,0.9752,0.9764,0.0012,0.3486
2,FINDHRLIST,4,0.9750,0.9764,0.0014,0.0253
3,FINDHRLIST,6,0.9728,0.9748,0.0020,0.0732
4,FINDHRLIST,10,0.9741,0.9769,0.0027,0.0022
5,FINDHR,1,0.9521,0.9535,0.0014,0.7718
6,FINDHR,2,0.9545,0.9602,0.0057,0.0136
7,FINDHR,4,0.9539,0.9602,0.0063,0.1031
8,FINDHR,6,0.9567,0.9648,0.0080,0.0022
9,FINDHR,10,0.9516,0.9593,0.0077,0.0017


In [9]:
print("guadagno medio: %+.4f" % principale["gain"].mean())
print("positivo in %d combinazioni su %d" % ((principale["gain"] > 0).sum(), len(principale)))
print("significativo in %d su %d" % ((principale["p_wilcoxon"] < 0.05).sum(), len(principale)))

guadagno medio: +0.0038
positivo in 10 combinazioni su 10
significativo in 6 su 10


Il vantaggio c'e' in tutte le combinazioni, ma e' piccolo e in quattro casi su dieci
non si distingue dal rumore. Va letto insieme alla sezione 6, dove si vede che sopra di
noi lo spazio disponibile e' comunque poco.

## 5. Cosa pesa di piu': la distanza o la correzione dentro la cella

Partendo dallo stesso modello di riferimento, che e' RTR con il PDT e il kNN, si
possono fare due interventi diversi. Il primo cambia la distanza e lascia il kNN. Il
secondo sostituisce tutta la correzione dentro la cella con la foresta, che come
abbiamo detto non e' interpretabile.

In [10]:
pezzi = pd.read_csv("distanza_o_aggregazione.csv")
pezzi[["dataset", "phi", "riferimento", "gain_distanza", "p_distanza",
       "gain_aggregazione", "p_aggregazione"]]

,dataset,phi,riferimento,gain_distanza,p_distanza,gain_aggregazione,p_aggregazione
0,FINDHRLIST,1,0.9770,0.0014,0.0161,0.0036,0.0093
1,FINDHRLIST,2,0.9752,0.0012,0.3486,0.0054,0.0002
2,FINDHRLIST,4,0.9750,0.0014,0.0253,0.0066,0.0000
3,FINDHRLIST,6,0.9728,0.0020,0.0732,0.0084,0.0000
4,FINDHRLIST,10,0.9741,0.0027,0.0022,0.0085,0.0000
5,FINDHR,1,0.9521,0.0014,0.7718,0.0055,0.0130
6,FINDHR,2,0.9545,0.0057,0.0136,0.0093,0.0004
7,FINDHR,4,0.9539,0.0063,0.1031,0.0124,0.0005
8,FINDHR,6,0.9567,0.0080,0.0022,0.0132,0.0001
9,FINDHR,10,0.9516,0.0077,0.0017,0.0136,0.0003


In [11]:
print("cambiando la distanza:                   %+.4f" % pezzi["gain_distanza"].mean())
print("sostituendo la correzione nella cella:   %+.4f" % pezzi["gain_aggregazione"].mean())
print()
print("la seconda rende di piu' in %d casi su %d" % (
    (pezzi["gain_aggregazione"] > pezzi["gain_distanza"]).sum(), len(pezzi)))

cambiando la distanza:                   +0.0038
sostituendo la correzione nella cella:   +0.0086

la seconda rende di piu' in 10 casi su 10


La seconda modifica rende piu' del doppio della prima. Attenzione pero' a come si
legge: i due interventi non sono simmetrici, perche' il secondo cambia piu' cose del
primo. La conclusione corretta non e' che l'aggregatore conta piu' della distanza, ma
che conta piu' quello che succede dentro la cella rispetto a quale distanza si usa per
scegliere i vicini. E il modo per guadagnare quel margine, oggi, non e' interpretabile.

## 6. Dove siamo, fra il caso e i modelli non interpretabili

Per capire se un guadagno di qualche millesimo e' molto o poco serve sapere quanto
spazio c'e'. Questa tabella mette insieme il ranking casuale, i modelli interpretabili
e quelli non interpretabili usati come limite superiore.

In [12]:
scala = pd.read_csv("scala_dei_modelli.csv")
scala.head(10)

,dataset,phi,casuale,solo r(x),euclidea in foglia,RTR (PDT),RTRwRuleCard (GAM),kNN,LambdaMART,foresta in foglia,foresta sul gruppo,boosting sul gruppo
0,FINDHRLIST,1,0.5142 ± 0.0121,0.9726 ± 0.0002,0.9787 ± 0.0002,0.9770 ± 0.0003,0.9784 ± 0.0004,0.9458 ± 0.0000,0.9184 ± 0.0000,0.9806 ± 0.0002,0.9822 ± 0.0001,0.9822 ± 0.0000
1,FINDHRLIST,2,0.5190 ± 0.0086,0.9648 ± 0.0002,0.9770 ± 0.0003,0.9752 ± 0.0002,0.9764 ± 0.0007,0.9508 ± 0.0000,0.9642 ± 0.0000,0.9807 ± 0.0001,0.9799 ± 0.0001,0.9833 ± 0.0000
2,FINDHRLIST,4,0.5219 ± 0.0109,0.9640 ± 0.0001,0.9760 ± 0.0001,0.9750 ± 0.0001,0.9764 ± 0.0002,0.9570 ± 0.0000,0.9827 ± 0.0000,0.9816 ± 0.0001,0.9781 ± 0.0003,0.9813 ± 0.0000
3,FINDHRLIST,6,0.5214 ± 0.0169,0.9597 ± 0.0000,0.9744 ± 0.0001,0.9728 ± 0.0002,0.9748 ± 0.0003,0.9620 ± 0.0000,0.9833 ± 0.0000,0.9811 ± 0.0003,0.9791 ± 0.0003,0.9811 ± 0.0000
4,FINDHRLIST,10,0.5216 ± 0.0072,0.9633 ± 0.0000,0.9760 ± 0.0000,0.9741 ± 0.0000,0.9769 ± 0.0004,0.9581 ± 0.0000,0.9852 ± 0.0000,0.9827 ± 0.0001,0.9806 ± 0.0002,0.9832 ± 0.0000
5,FINDHR,1,0.6571 ± 0.0114,0.9490 ± 0.0012,0.9528 ± 0.0015,0.9521 ± 0.0015,0.9535 ± 0.0018,0.9329 ± 0.0000,0.9441 ± 0.0000,0.9576 ± 0.0015,0.9735 ± 0.0002,0.9630 ± 0.0000
6,FINDHR,2,0.6602 ± 0.0042,0.9541 ± 0.0002,0.9606 ± 0.0004,0.9545 ± 0.0005,0.9602 ± 0.0011,0.9388 ± 0.0000,0.9649 ± 0.0000,0.9638 ± 0.0008,0.9736 ± 0.0008,0.9734 ± 0.0000
7,FINDHR,4,0.6663 ± 0.0108,0.9506 ± 0.0003,0.9595 ± 0.0003,0.9539 ± 0.0006,0.9602 ± 0.0006,0.9516 ± 0.0000,0.9737 ± 0.0000,0.9662 ± 0.0008,0.9760 ± 0.0009,0.9782 ± 0.0000
8,FINDHR,6,0.6614 ± 0.0140,0.9548 ± 0.0001,0.9640 ± 0.0001,0.9567 ± 0.0002,0.9648 ± 0.0007,0.9513 ± 0.0000,0.9766 ± 0.0000,0.9699 ± 0.0009,0.9790 ± 0.0004,0.9822 ± 0.0000
9,FINDHR,10,0.6608 ± 0.0063,0.9496 ± 0.0000,0.9587 ± 0.0000,0.9516 ± 0.0004,0.9593 ± 0.0009,0.9575 ± 0.0000,0.9763 ± 0.0000,0.9651 ± 0.0011,0.9769 ± 0.0004,0.9843 ± 0.0000


In [13]:
def valore(x):
    """Le celle sono scritte come media piu' o meno deviazione standard."""
    return float(str(x).split("\u00b1")[0].strip())


NON_INTERPRETABILI = ["foresta in foglia", "foresta sul gruppo", "boosting sul gruppo", "LambdaMART"]

distanze, coperture = [], []
for _, r in scala.iterrows():
    noi = valore(r["RTRwRuleCard (GAM)"])
    caso = valore(r["casuale"])
    tetto = max(valore(r[c]) for c in NON_INTERPRETABILI)
    distanze.append(tetto - noi)
    coperture.append((noi - caso) / (tetto - caso) * 100)

print("distanza media dal migliore non interpretabile: %.4f" % (sum(distanze) / len(distanze)))
print("quota della scala coperta: dal %.1f%% al %.1f%%" % (min(coperture), max(coperture)))

distanza media dal migliore non interpretabile: 0.0128
quota della scala coperta: dal 92.3% al 99.2%


Restando interpretabili copriamo quasi tutta la distanza fra il caso e il miglior
modello non interpretabile. Questo spiega perche' le differenze fra PDT e GAM sono
piccole in assoluto: sopra di noi c'e' poco spazio.

## 7. Cosa abbiamo provato che non funziona

Non tutte le modifiche aiutano, e i risultati negativi sono informativi quanto quelli
positivi. Tutte queste sono misurate a cinque semi sulle stesse dieci combinazioni e
contro lo stesso riferimento.

In [14]:
varianti = pd.read_csv("varianti_del_modello.csv")
riassunto = varianti.groupby("variante", sort=False).agg(
    media=("gain", "mean"),
    positiva=("gain", lambda s: int((s > 0).sum())),
    combinazioni=("gain", "size"),
)
riassunto["significativa"] = varianti.groupby("variante", sort=False)["p"].apply(lambda s: int((s < 0.05).sum()))
riassunto.round(4)

,media,positiva,combinazioni,significativa
variante,,,,
kNN pesato per distanza,-0.0060,0,10,8
minimo 6 documenti per foglia,-0.0009,4,10,2
minimo 10 documenti per foglia,-0.0035,2,10,2
foresta nella cella (non interpretabile),0.0086,10,10,10


Il kNN pesato per la distanza, invece che uniforme, peggiora in tutte e dieci le
combinazioni. Il motivo si capisce guardando le celle: sono piccole, e il numero di
vicini richiesto e' quasi uguale al numero di documenti disponibili, quindi il kNN
prende comunque quasi tutta la cella e pesare i valori della distanza aggiunge solo
rumore. Il vincolo sul numero minimo di documenti per foglia non sposta niente.

Messo insieme alla sezione 5, il quadro e' questo: nessuna modifica interpretabile
migliora il secondo stadio, e l'unica cosa che guadagna non e' interpretabile. Lo
spazio aperto e' un aggregatore che guadagni restando leggibile.

## 8. La scelta degli iperparametri

Per ogni gruppo di query si provano tutte le 144 configurazioni della griglia con una
convalida sui documenti di ogni query, e si tiene la migliore. La procedura e' identica
per i due modelli.

Con tre fold, che e' il valore di partenza del codice, la procedura fa danno dove i
gruppi sono piccoli. Rifacendola con dieci fold il danno sparisce.

In [15]:
fold = pd.read_csv("tre_fold_contro_dieci_test.csv")
fold[["dataset", "phi", "fissa", "3 fold - fissa", "p 3 fold - fissa",
      "10 fold - fissa", "p 10 fold - fissa"]]

,dataset,phi,fissa,3 fold - fissa,p 3 fold - fissa,10 fold - fissa,p 10 fold - fissa
0,FINDHR,1,0.9521,-0.0177,0.0012,-0.0040,0.3647
1,FINDHR,2,0.9545,-0.0129,0.0249,-0.0005,0.6660
2,FINDHR,4,0.9539,-0.0008,0.7095,0.0033,0.2004
3,FINDHR,6,0.9567,0.0046,0.0559,0.0034,0.1026
4,FINDHR,10,0.9516,0.0116,0.0068,0.0080,0.0240
5,FINDHRLIST,1,0.9770,-0.0003,0.8016,0.0010,0.6749
6,FINDHRLIST,2,0.9752,0.0051,0.0831,0.0043,0.2561
7,FINDHRLIST,4,0.9750,0.0046,0.0102,0.0016,0.5360
8,FINDHRLIST,6,0.9728,0.0075,0.0019,0.0070,0.0018
9,FINDHRLIST,10,0.9741,0.0054,0.0052,0.0060,0.0000


In [16]:
peggiora_3 = ((fold["3 fold - fissa"] < 0) & (fold["p 3 fold - fissa"] < 0.05)).sum()
peggiora_10 = ((fold["10 fold - fissa"] < 0) & (fold["p 10 fold - fissa"] < 0.05)).sum()
migliora_10 = ((fold["10 fold - fissa"] > 0) & (fold["p 10 fold - fissa"] < 0.05)).sum()

print("con tre fold la scelta per gruppo e' significativamente peggiore in %d casi su %d" % (peggiora_3, len(fold)))
print("con dieci fold e' significativamente peggiore in %d casi su %d" % (peggiora_10, len(fold)))
print("con dieci fold e' significativamente migliore in %d casi su %d" % (migliora_10, len(fold)))

con tre fold la scelta per gruppo e' significativamente peggiore in 2 casi su 10
con dieci fold e' significativamente peggiore in 0 casi su 10
con dieci fold e' significativamente migliore in 3 casi su 10


Resta pero' un dato che conviene tenere presente: cambiando solo il numero di fold, la
configurazione scelta cambia in 90 gruppi su 100 su un dataset e in 99 su 100
sull'altro. Una procedura che ribalta quasi tutte le sue scelte al cambiare di un
dettaglio della convalida sta in buona parte scegliendo rumore, quindi i guadagni
stimati in validazione vanno presi come ottimistici.

In [17]:
scelte = pd.read_csv("tre_fold_contro_dieci.csv")
scelte[["dataset", "phi", "gruppi", "profondita_media_3fold", "profondita_media_10fold", "config_diversa"]]

,dataset,phi,gruppi,profondita_media_3fold,profondita_media_10fold,config_diversa
0,FINDHR,1,100,3.34,3.74,90
1,FINDHR,2,50,4.20,4.48,44
2,FINDHR,4,25,5.44,4.96,23
3,FINDHR,6,17,5.76,5.41,15
4,FINDHR,10,10,7.00,6.00,8
5,FINDHRLIST,1,100,4.56,5.24,99
6,FINDHRLIST,2,50,5.08,5.52,48
7,FINDHRLIST,4,25,5.36,5.52,25
8,FINDHRLIST,6,17,5.65,5.65,16
9,FINDHRLIST,10,10,6.60,5.80,9


## 9. Quanto costa

I tempi sono presi dalle esecuzioni pubblicate. Il rapporto non e' costante: cresce
al diminuire del numero di query per modello, perche' con gruppi piccoli le celle sono
tante e la GAM va costruita in ognuna. La GAM guadagna poco e costa molto, ed e' un
punto da tenere presente quando si decide se usarla.

In [18]:
import json


def tempo_mediano(modello, dataset, phi):
    # tempo di addestramento mediano fra i semi, per una combinazione
    tempi = []
    for f in (RUN / dataset / modello / f"phi{phi}").glob("seed*/metriche.json"):
        tempi.append(json.loads(f.read_text(encoding="utf-8"))["tempo_fit_s"])
    tempi.sort()
    return tempi[len(tempi) // 2]


rapporti = []
for dataset in ["FINDHR", "FINDHRLIST"]:
    for phi in [1, 2, 4, 6, 10]:
        pdt_s = tempo_mediano("rtr", dataset, phi)
        gam_s = tempo_mediano("rtrwrulecard", dataset, phi)
        rapporti.append(gam_s / pdt_s)
        print("%-11s query per modello %-3d PDT %6.1f s | GAM %7.1f s | %5.1f volte" % (
            dataset, phi, pdt_s, gam_s, gam_s / pdt_s))

print()
print("rapporto: da %.1f a %.1f volte, media %.1f" % (
    min(rapporti), max(rapporti), sum(rapporti) / len(rapporti)))

FINDHR      query per modello 1   PDT   31.0 s | GAM   746.9 s |  24.1 volte
FINDHR      query per modello 2   PDT   14.7 s | GAM   429.9 s |  29.2 volte
FINDHR      query per modello 4   PDT   11.7 s | GAM   264.3 s |  22.6 volte
FINDHR      query per modello 6   PDT   11.6 s | GAM   197.5 s |  17.0 volte


FINDHR      query per modello 10  PDT   10.8 s | GAM   140.7 s |  13.0 volte
FINDHRLIST  query per modello 1   PDT   34.0 s | GAM   447.2 s |  13.2 volte
FINDHRLIST  query per modello 2   PDT   17.6 s | GAM   278.7 s |  15.9 volte
FINDHRLIST  query per modello 4   PDT   14.7 s | GAM   166.8 s |  11.4 volte
FINDHRLIST  query per modello 6   PDT   13.9 s | GAM   122.0 s |   8.8 volte


FINDHRLIST  query per modello 10  PDT   12.4 s | GAM    88.4 s |   7.1 volte

rapporto: da 7.1 a 29.2 volte, media 16.2


## 10. Cosa resta aperto

La tabella a configurazione scelta per gruppo e' stata prodotta con la selezione a tre
fold, cioe' con la procedura che ora sappiamo difettosa. Per rifarla in modo pulito
serve rifare la selezione a dieci fold anche per il modello additivo, che costa circa
dieci volte quella con il PDT. Finche' non e' fatta, quella riga va considerata
provvisoria.

Resta poi la domanda aperta piu' interessante: un aggregatore che guadagni restando
interpretabile. Sappiamo che il margine esiste, perche' la foresta lo prende, e
sappiamo che nessuna delle modifiche interpretabili provate finora lo prende.